# Notebook 05: So sánh Toàn diện và Quyết định Chọn Mô hình (Model Comparison & Final Selection)

**Người phụ trách chính:** TV3 (Bộ đánh giá & Bảng so sánh). Toàn nhóm rà soát rò rỉ dữ liệu.

### Mục tiêu nghiên cứu:
1. Tải kết quả kiểm định chéo 5-fold CV đã khóa của cả 4 mô hình: Baseline Dummy, Decision Tree, Logistic Regression, Random Forest.
2. Áp dụng quy tắc lựa chọn Champion theo đúng Plan 03 (ưu tiên F1 lớp 1, áp dụng tie-breaker khi chênh lệch < 0.005).
3. Kiểm chứng hiệu năng trên tập kiểm thử độc lập (Test Set: N=1.409), vẽ đồ thị so sánh ROC, Precision-Recall và phân tích ma trận nhầm lẫn.
4. Tiến hành phân tích lỗi định tính (Error Analysis) trên các ca TP, FN, FP.


In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
from telco_churn.inference import load_model_bundle, predict_records
from telco_churn.evaluation import (
    plot_confusion_matrix, plot_roc_curves, plot_pr_curves, plot_model_comparison
)

sns.set_theme(style='whitegrid')


## 1. Đọc tệp tổng hợp Cross-Validation và Báo cáo Quyết định Chọn Mô hình


In [ ]:
with open('../artifacts/models/model_001/cv_summary.json', 'r', encoding='utf-8') as f:
    cv_summary = json.load(f)

rows = []
for k, v in cv_summary.items():
    rows.append({
        'Model': k,
        'CV F1 (Mean ± Std)': f"{v['mean_f1']:.4f} ± {v['std_f1']:.4f}",
        'CV Recall': f"{v['mean_recall']:.4f}",
        'CV Precision': f"{v['mean_precision']:.4f}",
        'CV ROC-AUC': f"{v['mean_roc_auc']:.4f}",
        'Fit Time (s)': f"{v.get('mean_fit_time', 0.0):.2f}s"
    })
df_cv = pd.DataFrame(rows)
print('=== BẢNG SO SÁNH 5-FOLD CROSS-VALIDATION (DEVELOPMENT SET) ===')
print(df_cv.to_string(index=False))


## 2. Kiểm chứng Hiệu năng Độc lập trên Tập Test (Test Set: N=1.409)


In [ ]:
with open('../artifacts/models/model_001/test_metrics.json', 'r', encoding='utf-8') as f:
    test_data = json.load(f)

print(f"Champion Model: {test_data['champion_model'].upper()}")
champ_met = test_data['champion_test_metrics']
print(f"Test Accuracy:  {champ_met['accuracy']:.4f}")
print(f"Test F1-Score:  {champ_met['f1']:.4f}")
print(f"Test Recall:    {champ_met['recall']:.4f}")
print(f"Test Precision: {champ_met['precision']:.4f}")
print(f"Test ROC-AUC:   {champ_met['roc_auc']:.4f}")
print(f"Test PR-AUC:    {champ_met['average_precision']:.4f}")


## 3. Trực quan hóa Biểu đồ So sánh Chỉ số CV vs. Test


In [ ]:
fig = plot_model_comparison(cv_summary, test_data['all_models_test_metrics'])
plt.show()


## 4. Đánh giá Ma trận Nhầm lẫn và Báo cáo Nghiệp vụ
- Mô hình Champion `Logistic Regression` nhận diện thành công **294 / 374** khách hàng thực tế rời bỏ (đạt độ nhạy **78.6%**).
- Số lượng ca âm tính giả (False Negatives) được kìm hãm ở mức 80 khách hàng, giúp doanh nghiệp chủ động bảo vệ nguồn doanh thu thuê bao.
- Toàn bộ quy trình từ dữ liệu, tiền xử lý, kiểm định chéo đến đánh giá kiểm thử đều tuân thủ 100% nguyên tắc không rò rỉ dữ liệu.
